In [ ]:
import json
from pathlib import Path

base = Path.home() / "POPE/output/coco_original"
out_dir = Path.home() / "learn-to-steer/data/pope_descriptive"
out_dir.mkdir(parents=True, exist_ok=True)


files = {
    "random": base / "coco_pope_random.json",
    "popular": base / "coco_pope_popular.json",
    "adversarial": base / "coco_pope_adversarial.json",
}

merged = []

for subset, path in files.items():
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            ex = json.loads(line)
            merged.append({
                "filename": ex["image"],
                "instruction": ex["text"] + " Answer with just one word.",
                "response": ex["label"],
                "subset": subset,
            })

out_path = out_dir / "annotations.json"
with open(out_path, "w") as f:
    json.dump(merged, f, indent=2)

print(f"Saved {len(merged)} examples to {out_path}")

Saved 9000 examples to /research/hal-afsharim/learn-to-steer/data/pope/train/annotations.json


In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import torch

def set_seed(seed_value=0):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    
seed = 0
set_seed(seed)

# Change these paths if needed
pope_root = Path("/research/hal-afsharim/POPE/output/coco_original")
out_root = Path("/research/hal-afsharim/POPE/pope_descriptive")
coco_val2014 = Path("/research/hal-afsharim/learn-to-steer/data/coco/val2014")

files = {
    "random": pope_root / "coco_pope_random.json",
    "popular": pope_root / "coco_pope_popular.json",
    "adversarial": pope_root / "coco_pope_adversarial.json",
}

splits = {
    "train": [],
    "val": [],
    "test": [],
}

for subset, path in files.items():
    subset_data = []

    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            ex = json.loads(line)  # POPE files are JSONL
            subset_data.append({
                "filename": ex["image"],
                "instruction": ex["text"] + " Answer with just one word.",
                "response": ex["label"],
                "subset": subset,
            })

    random.shuffle(subset_data)

    n = len(subset_data)
    n_train = int(0.7 * n)
    n_val = int(0.1 * n)
    n_test = n - n_train - n_val

    splits["train"].extend(subset_data[:n_train])
    splits["val"].extend(subset_data[n_train:n_train + n_val])
    splits["test"].extend(subset_data[n_train + n_val:])

# Optional: shuffle within each final split so subsets are mixed
for split_name in splits:
    random.shuffle(splits[split_name])

for split_name, split_data in splits.items():
    split_dir = out_root / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    with open(split_dir / "annotations.json", "w") as f:
        json.dump(split_data, f, indent=2)

    images_link = split_dir / "images"
    if images_link.is_symlink() or images_link.exists():
        images_link.unlink()
    images_link.symlink_to(coco_val2014)

    print(f"{split_name}: {len(split_data)} samples -> {split_dir / 'annotations.json'}")

print("Per-split subset counts:")
for split_name, split_data in splits.items():
    counts = {}
    for x in split_data:
        counts[x["subset"]] = counts.get(x["subset"], 0) + 1
    print(split_name, counts)

print(f"Total: {sum(len(v) for v in splits.values())}")

train: 37800 samples -> /research/hal-afsharim/POPE/pope_split_more_data/train/annotations.json
val: 5400 samples -> /research/hal-afsharim/POPE/pope_split_more_data/val/annotations.json
test: 10800 samples -> /research/hal-afsharim/POPE/pope_split_more_data/test/annotations.json
Per-split subset counts:
train {'adversarial': 12600, 'popular': 12600, 'random': 12600}
val {'popular': 1800, 'random': 1800, 'adversarial': 1800}
test {'random': 3600, 'popular': 3600, 'adversarial': 3600}
Total: 54000


In [2]:
import json
import random
from pathlib import Path

import numpy as np
import torch

def set_seed(seed_value=0):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed = 1
set_seed(seed)

# Paths
pope_root = Path("/research/hal-afsharim/POPE/output/coco_original")
out_root = Path("/research/hal-afsharim/POPE/pope_descriptive")
coco_val2014 = Path("/research/hal-afsharim/learn-to-steer/data/coco/val2014")

files = {
    "random": pope_root / "coco_pope_random.json",
    "popular": pope_root / "coco_pope_popular.json",
    "adversarial": pope_root / "coco_pope_adversarial.json",
}

# Collect every image filename that appears anywhere in POPE
pope_images = set()
for path in files.values():
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            ex = json.loads(line)
            pope_images.add(ex["image"])

# All images in coco val2014
all_val_images = sorted(p.name for p in coco_val2014.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

# Exclude any image that appears in POPE
candidates = [name for name in all_val_images if name not in pope_images]
print(f"POPE images: {len(pope_images)}, val2014 images: {len(all_val_images)}, candidates: {len(candidates)}")

# Sample 500
n_sample = 500
assert len(candidates) >= n_sample, f"Not enough non-POPE images: {len(candidates)} < {n_sample}"
sampled = random.sample(candidates, n_sample)

descriptive_data = [
    {
        "filename": name,
        "instruction": "Describe this image in detail.",
        "response": "",
        "subset": "descriptive",
    }
    for name in sampled
]

# Write out
split_dir = out_root / "descriptive"
split_dir.mkdir(parents=True, exist_ok=True)

with open(split_dir / "annotations.json", "w") as f:
    json.dump(descriptive_data, f, indent=2)

images_link = split_dir / "images"
if images_link.is_symlink() or images_link.exists():
    images_link.unlink()
images_link.symlink_to(coco_val2014)

print(f"descriptive: {len(descriptive_data)} samples -> {split_dir / 'annotations.json'}")

# Sanity check: no overlap with POPE
overlap = set(sampled) & pope_images
print(f"Overlap with POPE: {len(overlap)} (should be 0)")


POPE images: 500, val2014 images: 40504, candidates: 40004
descriptive: 500 samples -> /research/hal-afsharim/POPE/pope_descriptive/descriptive/annotations.json
Overlap with POPE: 0 (should be 0)
